In [2]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = display = None


pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)

if display is not None:
    display(HTML("""
    <style>
    .dataframe th, .dataframe td {
        white-space: pre-wrap;
        word-break: break-word;
        max-width: 420px;
    }
    </style>
    """))


PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_DIR / "data_raw"
META_DIR = PROJECT_DIR / "metadata"

B1_FILE = META_DIR / "MiD2023_Codepläne_B1_Standard_v1.1.xlsx"

SUBFILES = {
    "Haushalte": DATA_DIR / "MiD2023_Haushalte.csv",
    "Wege": DATA_DIR / "MiD2023_Wege.csv",
    "Autos": DATA_DIR / "MiD2023_Autos.csv",
    "Personen": DATA_DIR / "MiD2023_Personen.csv",
}

CSV_READ_OPTIONS = {
    "sep": None,
    "engine": "python",
    "dtype": str,
    "encoding": "utf-8-sig",
}

csv_files = sorted(DATA_DIR.glob("*.csv"))
missing_csv = [name for name, path in SUBFILES.items() if not path.exists()]

print(f"Found {len(csv_files)} CSV files")
print("B1 file exists:", B1_FILE.exists())
print("Configured helper files:", ", ".join(SUBFILES))
if missing_csv:
    print("Missing configured CSV files:", ", ".join(missing_csv))


Found 7 CSV files
B1 file exists: True
Configured helper files: Haushalte, Wege, Autos, Personen


In [3]:
CODE_COLUMNS = [
    "position", "variable", "variable_label",
    "value", "value_label", "measure_level", "format"
]

FFILL_COLUMNS = ["variable", "variable_label", "measure_level", "format"]


def read_codeplan(sheet_name):
    """Read and normalize one B1 codeplan sheet."""
    df = pd.read_excel(
        B1_FILE,
        sheet_name=sheet_name,
        header=1,
        dtype="string",
    ).copy()
    df.columns = CODE_COLUMNS
    df.insert(0, "source", sheet_name)

    # Fill down variable-level information so value rows stay searchable.
    for col in FFILL_COLUMNS:
        df[f"{col}_ffill"] = df[col].ffill()

    return df


def read_raw_columns(source):
    """Read only the CSV header for one configured subfile."""
    return list(pd.read_csv(SUBFILES[source], nrows=0, **CSV_READ_OPTIONS).columns)


codebooks = {source: read_codeplan(source) for source in SUBFILES}
code_all = pd.concat(codebooks.values(), ignore_index=True)
raw_columns = {source: read_raw_columns(source) for source in SUBFILES}

print("Loaded codebook rows:", len(code_all))
for source, columns in raw_columns.items():
    print(f"{source}: {len(codebooks[source])} code rows, {len(columns)} CSV columns")


Loaded codebook rows: 4210
Haushalte: 427 code rows, 79 CSV columns
Wege: 1341 code rows, 209 CSV columns
Autos: 469 code rows, 77 CSV columns
Personen: 1973 code rows, 261 CSV columns


In [4]:
DISPLAY_COLS = [
    "source", "position", "variable_ffill", "variable_label_ffill",
    "value", "value_label", "measure_level_ffill", "format_ffill"
]

SUMMARY_COLS = [
    "source", "position", "variable_ffill", "variable_label_ffill",
    "measure_level_ffill", "format_ffill", "n_value_labels", "in_csv"
]

FIELD_ALIASES = {
    "source": "source",
    "sheet": "source",
    "datei": "source",
    "position": "position",
    "variable": "variable_ffill",
    "Variable": "variable_ffill",
    "variablenlabel": "variable_label_ffill",
    "Variablenlabel": "variable_label_ffill",
    "variable_label": "variable_label_ffill",
    "wert": "value",
    "Wert": "value",
    "wertelabel": "value_label",
    "Wertelabel": "value_label",
    "messniveau": "measure_level_ffill",
    "Messniveau": "measure_level_ffill",
    "format": "format_ffill",
    "Format": "format_ffill",
}


def _as_list(value):
    if value is None:
        return None
    if isinstance(value, str):
        return [value]
    return list(value)


def _resolve_field(field):
    if field in code_all.columns:
        return field
    if field in FIELD_ALIASES:
        return FIELD_ALIASES[field]
    raise ValueError(f"Unknown field: {field}. Valid examples: Variable, Variablenlabel, Wertelabel")


def _subset_sources(df, sources=None):
    selected = _as_list(sources)
    if selected is None:
        return df

    unknown = sorted(set(selected) - set(SUBFILES))
    if unknown:
        raise ValueError(f"Unknown source(s): {unknown}. Use one of {list(SUBFILES)}")

    return df.loc[df["source"].isin(selected)]


def _count_value_labels(row):
    mask = (
        code_all["source"].eq(row["source"])
        & code_all["variable_ffill"].eq(row["variable_ffill"])
        & code_all["value"].notna()
    )
    return int(mask.sum())


def _exists_in_csv(row):
    return row["variable_ffill"] in raw_columns.get(row["source"], [])


def search_codebook(term, field="Variablenlabel", sources=None, case=False, regex=False, one_row_per_variable=True):
    """
    Search codebook rows like Excel contains-filter.

    Examples:
    search_codebook("Elektrofahrräder", field="Variablenlabel")
    search_codebook("Auto", field="Wertelabel", sources=["Haushalte", "Personen"], one_row_per_variable=False)
    """
    field_col = _resolve_field(field)
    df = _subset_sources(code_all, sources=sources)
    mask = df[field_col].astype("string").str.contains(term, case=case, regex=regex, na=False)
    result = df.loc[mask].copy()

    if one_row_per_variable:
        result = (
            result.sort_values(["source", "position", "variable_ffill"])
            .drop_duplicates(["source", "variable_ffill"])
        )
        result["n_value_labels"] = result.apply(_count_value_labels, axis=1)
        result["in_csv"] = result.apply(_exists_in_csv, axis=1)
        return result.loc[:, SUMMARY_COLS].reset_index(drop=True)

    return result.loc[:, DISPLAY_COLS].reset_index(drop=True)


def find_variables(term, field="Variablenlabel", sources=None, case=False, regex=False):
    """Return one row per matched variable, useful for choosing variables for analysis."""
    return search_codebook(
        term,
        field=field,
        sources=sources,
        case=case,
        regex=regex,
        one_row_per_variable=True,
    )


def find_rows(term, field="Variablenlabel", sources=None, case=False, regex=False):
    """Return all matched codebook rows, including value-label rows."""
    return search_codebook(
        term,
        field=field,
        sources=sources,
        case=case,
        regex=regex,
        one_row_per_variable=False,
    )


def show_code(var, sources=None, case=False):
    """Show all value labels and metadata for a variable across selected subfiles."""
    df = _subset_sources(code_all, sources=sources)
    variable = str(var)
    variables = df["variable_ffill"].astype("string")

    if case:
        mask = variables.eq(variable)
    else:
        mask = variables.str.lower().eq(variable.lower())

    return df.loc[mask, DISPLAY_COLS].reset_index(drop=True)


def list_variables(sources=None):
    """List variables available in selected subfiles."""
    df = _subset_sources(code_all, sources=sources)
    result = df.loc[df["variable"].notna()].copy()
    result["n_value_labels"] = result.apply(_count_value_labels, axis=1)
    result["in_csv"] = result.apply(_exists_in_csv, axis=1)
    return result.loc[:, SUMMARY_COLS].reset_index(drop=True)


def csv_columns(sources=None):
    """List CSV columns for selected subfiles."""
    selected = _as_list(sources) or list(SUBFILES)
    return pd.DataFrame(
        [(source, column) for source in selected for column in raw_columns[source]],
        columns=["source", "column"],
    )


In [5]:
PROCESSED_DIR = PROJECT_DIR / "data_processed"

SELECTED_RAW_FILES = {
    "Haushalte": PROCESSED_DIR / "hh_selected_raw.csv",
    "Personen": PROCESSED_DIR / "persons_selected_raw.csv",
    "Wege": PROCESSED_DIR / "trips_selected_raw.csv",
    "Autos": PROCESSED_DIR / "cars_selected_raw.csv",
}

SELECTED_RAW_READ_OPTIONS = {
    "dtype": "string",
    "keep_default_na": False,
    "na_values": [],
}


def _selected_raw_sources(sources=None):
    if sources is None:
        return list(SELECTED_RAW_FILES)
    selected = [sources] if isinstance(sources, str) else list(sources)
    unknown = sorted(set(selected) - set(SELECTED_RAW_FILES))
    if unknown:
        raise ValueError(f"Unknown selected raw source(s): {unknown}. Use one of {list(SELECTED_RAW_FILES)}")
    return selected


def _selected_raw_path(source):
    path = SELECTED_RAW_FILES[source]
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run scripts/01_extract_selected_raw.py before checking selected raw CSVs."
        )
    return path


def _selected_raw_columns(source):
    return list(pd.read_csv(_selected_raw_path(source), nrows=0).columns)


def _selected_raw_row_count(source, chunk_size=200_000):
    columns = _selected_raw_columns(source)
    if not columns:
        return 0
    reader = pd.read_csv(
        _selected_raw_path(source),
        usecols=[columns[0]],
        chunksize=chunk_size,
        **SELECTED_RAW_READ_OPTIONS,
    )
    return sum(len(chunk) for chunk in reader)


def selected_raw_overview(sources=None):
    """Return file, shape, and path checks for selected raw CSV outputs."""
    rows = []
    for source in _selected_raw_sources(sources):
        path = _selected_raw_path(source)
        columns = _selected_raw_columns(source)
        rows.append({
            "source": source,
            "rows": _selected_raw_row_count(source),
            "columns": len(columns),
            "file": path.name,
            "path": str(path),
        })
    return pd.DataFrame(rows)


def selected_raw_column_descriptions(source):
    """Return selected raw columns with codebook labels, measure levels, and formats."""
    columns = pd.DataFrame({"column": _selected_raw_columns(source)})
    meta = (
        code_all.loc[
            code_all["source"].eq(source) & code_all["variable_ffill"].notna(),
            ["variable_ffill", "variable_label_ffill", "measure_level_ffill", "format_ffill"],
        ]
        .drop_duplicates("variable_ffill")
        .rename(columns={
            "variable_ffill": "column",
            "variable_label_ffill": "variable_label",
            "measure_level_ffill": "measure_level",
            "format_ffill": "format",
        })
    )
    result = columns.merge(meta, on="column", how="left")
    result.insert(0, "source", source)
    result["has_codebook_description"] = result["variable_label"].notna()
    return result



print("Configured selected raw files:", ", ".join(SELECTED_RAW_FILES))

# selected_raw_overview()
# selected_raw_column_descriptions("Haushalte")



Configured selected raw files: Haushalte, Personen, Wege, Autos


In [6]:
selected_raw_overview()

,source,rows,columns,file,path
0,Haushalte,218101,46,hh_selected_raw.csv,c:\Users\VWJ5RQ4\Documents\data\MiD internal\CSV\data_processed\hh_selected_raw.csv
1,Personen,420979,79,persons_selected_raw.csv,c:\Users\VWJ5RQ4\Documents\data\MiD internal\CSV\data_processed\persons_selected_raw.csv
2,Wege,1087393,135,trips_selected_raw.csv,c:\Users\VWJ5RQ4\Documents\data\MiD internal\CSV\data_processed\trips_selected_raw.csv
3,Autos,290930,24,cars_selected_raw.csv,c:\Users\VWJ5RQ4\Documents\data\MiD internal\CSV\data_processed\cars_selected_raw.csv


In [7]:
selected_raw_column_descriptions("Haushalte")

,source,column,variable_label,measure_level,format,has_codebook_description
0,Haushalte,H_ID,Haushalts-ID,Metrisch,F8,True
1,Haushalte,H_GEW,Gewichtungsfaktor Haushalte,Metrisch,F8.2,True
2,Haushalte,H_HOCH,Hochrechnungsfaktor Haushalte,Metrisch,F8.2,True
3,Haushalte,MODE,Erhebungsmethode,Nominal,F1,True
4,Haushalte,BASISAUF,Basis- oder Aufstockungsstichprobe,Nominal,F1,True
5,Haushalte,M_CAR,Modul - Fahrzeugmerkmale und Fahrzeugbesitz,Nominal,F1,True
6,Haushalte,H_ART,Art des Haushalts,Nominal,F1,True
7,Haushalte,H_GR,Haushaltsgröße,Metrisch,F2,True
8,Haushalte,hhgr_gr,Haushaltsgröße in Gruppen (1 bis 5+),Ordinal,F1,True
9,Haushalte,HP_ALTER_1,Alter - Person 1 (Angabe Personen-/HH-Interview),Metrisch,F3,True


In [8]:
selected_raw_column_descriptions("Personen")

,source,column,variable_label,measure_level,format,has_codebook_description
0,Personen,HP_ID,Haushalts-Personen-ID,Metrisch,F8,True
1,Personen,H_ID,Haushalts-ID,Metrisch,F8,True
2,Personen,P_ID,Personen-ID,Nominal,F1,True
3,Personen,P_GEW,Gewichtungsfaktor Personen,Metrisch,F8.2,True
4,Personen,P_HOCH,Hochrechnungsfaktor Personen,Metrisch,F8.2,True
...,...,...,...,...,...,...
74,Personen,pergrup2,verhaltenshomogene Personengruppen (9 Gruppen),Nominal,F2,True
75,Personen,seg_vm,Nutzungssegmente IV und ÖV,Nominal,F3,True
76,Personen,multimodal,multimodale Personengruppen (Basis Nutzungshäufigkeit VM in einer Woche),Nominal,F3,True
77,Personen,intermodal,Person hat intermodale Wege am Stichtag ja/nein (0-1-Codierung) (analog zu MiD2017),Nominal,F3,True


In [9]:
selected_raw_column_descriptions("Wege")

,source,column,variable_label,measure_level,format,has_codebook_description
0,Wege,HP_ID,Haushalts-Personen-ID,Metrisch,F8,True
1,Wege,H_ID,Haushalts-ID,Metrisch,F8,True
2,Wege,P_ID,Personen-ID,Nominal,F1,True
3,Wege,W_ID,Wege-ID (sortiert),Ordinal,F3,True
4,Wege,W_GEW,Gewichtungsfaktor Wege,Metrisch,F8.2,True
...,...,...,...,...,...,...
130,Wege,quali_nv_so,Qualität der Nahversorgung (Raumbezug: Adresse) am Startort des Weges,Ordinal,F3,True
131,Wege,quali_nv_zo,Qualität der Nahversorgung (Raumbezug: Adresse) am Zielort des Weges,Ordinal,F3,True
132,Wege,quali_opnv,Qualität des ÖPNV (Raumbezug: Adresse) am Wohnort,Ordinal,F2,True
133,Wege,quali_opnv_so,Qualität des ÖPNV (Raumbezug: Adresse) am Startort des Weges,Ordinal,F3,True


In [10]:
selected_raw_column_descriptions("Autos")

,source,column,variable_label,measure_level,format,has_codebook_description
0,Autos,H_ID,Haushalts-ID,Metrisch,F8,True
1,Autos,A_ID,Auto-ID,Nominal,F1,True
2,Autos,A_GEW,Gewichtungsfaktor Autos,Metrisch,F8.2,True
3,Autos,M_CAR,Modul - Fahrzeugmerkmale und Fahrzeugbesitz,Nominal,F1,True
4,Autos,H_ANZAUTO,Anzahl Autos im HH,Metrisch,F2,True
5,Autos,A_ANTRIEB,Antriebsart,Nominal,F2,True
6,Autos,antrieb,Antriebsart (zusammengefasst),Nominal,F2,True
7,Autos,A_HALTER,Halter/in (CATI/CAWI),Nominal,F3,True
8,Autos,A_BAUJ,Baujahr,Metrisch,F4,True
9,Autos,bauj_gr,Baujahr in Gruppen,Ordinal,F2,True


In [15]:
# Examples
# 1) Similar to Excel contains-filter: Variablenlabel contains "Elektrofahrräder".
find_variables("entfernung", field="Variablenlabel")

,source,position,variable_ffill,variable_label_ffill,measure_level_ffill,format_ffill,n_value_labels,in_csv
0,Autos,38,durchkm,durchschnittliche Entfernung der Fahrten am Stichtag [km] (Mittelwert wegkm_imp),Metrisch,F8.2,2,True
1,Autos,39,durchkm_gr,durchschnittliche Entfernung der Fahrten am Stichtag in Gruppen,Ordinal,F3,11,True
2,Autos,67,bus28,Luftlinienentfernung zur nächsten Bushaltestelle mit mind. 28 Abfahrten im ÖV an einem Werktag 2022 (Raumbezug: Geokoordinaten),Ordinal,F2,7,True
3,Autos,68,tram28,Luftlinienentfernung zur nächsten Straßenbahn-/U-Bahn-Haltestelle mit mind. 28 Abfahrten im ÖV an einem Werktag 2022 (Raumbezug: Geokoordinaten),Ordinal,F2,7,True
4,Autos,69,bahn28,Luftlinienentfernung zum nächsten Bahnhof mit mind. 28 Abfahrten im ÖV an einem Werktag 2022 (Raumbezug: Geokoordinaten),Ordinal,F2,7,True
5,Haushalte,70,bus28,Luftlinienentfernung zur nächsten Bushaltestelle mit mind. 28 Abfahrten im ÖV an einem Werktag 2022 (Raumbezug: Geokoordinaten),Ordinal,F2,7,True
6,Haushalte,71,tram28,Luftlinienentfernung zur nächsten Straßenbahn-/U-Bahn-Haltestelle mit mind. 28 Abfahrten im ÖV an einem Werktag 2022 (Raumbezug: Geokoordinaten),Ordinal,F2,7,True
7,Haushalte,72,bahn28,Luftlinienentfernung zum nächsten Bahnhof mit mind. 28 Abfahrten im ÖV an einem Werktag 2022 (Raumbezug: Geokoordinaten),Ordinal,F2,7,True
8,Personen,117,P_NEBEN_ENTF,Entfernung zum Nebenwohnsitz (CATI/CAWI),Ordinal,F3,16,True
9,Personen,137,P_ARB_ENTF,Entfernung zur Arbeitsstätte [km] (CATI/CAWI),Metrisch,F6.1,8,True


In [13]:
# 2) Show all value labels after you choose a variable.
show_code("zweck", sources="Wege")

,source,position,variable_ffill,variable_label_ffill,value,value_label,measure_level_ffill,format_ffill
0,Wege,49,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",1,Erreichen der Arbeitsstätte,Nominal,F2
1,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",2,dienstlich/geschäftlich,Nominal,F2
2,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",3,Erreichen der Ausbildungsstätte/Schule,Nominal,F2
3,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",4,Einkauf,Nominal,F2
4,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",5,private Erledigungen,Nominal,F2
5,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",6,Bringen/Holen/Begleiten von Personen,Nominal,F2
6,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",7,Freizeitaktivität,Nominal,F2
7,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",8,nach Hause,Nominal,F2
8,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",9,Rückweg vom vorherigen Weg,Nominal,F2
9,Wege,<NA>,zweck,"Wegezweck (zusammengefasst, nach Hause und Rückweg separat)",10,anderer Zweck,Nominal,F2
